In [1]:
%pip install shap


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import shap

In [3]:
df = pd.read_csv("loan_data.csv")
print(df.shape)

(52000, 27)


In [4]:
df .head()

,Applicant_ID,Gender,Age,Marital_Status,Dependents,Education,Employment_Status,Occupation_Type,Residential_Status,City/Town,...,Loan_Amount_Requested,Loan_Term,Loan_Purpose,Interest_Rate,Loan_Type,Co-Applicant,Bank_Account_History,Transaction_Frequency,Default_Risk,Loan_Approval_Status
0,1,Female,25,Married,2,Graduate,Employed,Business,Own,Urban,...,24535,209,Home,4.27,Secured,Yes,8,20,0.81,1
1,2,Male,36,Married,2,High School,Employed,Business,Own,Suburban,...,8288,33,Home,14.78,Unsecured,Yes,9,9,0.17,0
2,3,Male,43,Single,0,Postgraduate,Self-Employed,Freelancer,Own,Urban,...,10308,159,Vehicle,12.33,Secured,Yes,7,27,0.25,0
3,4,Female,28,Married,0,High School,Self-Employed,Freelancer,Rent,Suburban,...,33937,39,Personal,8.77,Secured,No,9,16,0.27,1
4,5,Female,32,Single,0,Graduate,Employed,Salaried,Rent,Suburban,...,23360,34,Home,9.04,Unsecured,No,1,17,0.32,1


In [5]:
df.isnull().sum()

Applicant_ID                  0
Gender                        0
Age                           0
Marital_Status                0
Dependents                    0
Education                     0
Employment_Status             0
Occupation_Type               0
Residential_Status            0
City/Town                     0
Annual_Income                 0
Monthly_Expenses              0
Credit_Score                  0
Existing_Loans                0
Total_Existing_Loan_Amount    0
Outstanding_Debt              0
Loan_History                  0
Loan_Amount_Requested         0
Loan_Term                     0
Loan_Purpose                  0
Interest_Rate                 0
Loan_Type                     0
Co-Applicant                  0
Bank_Account_History          0
Transaction_Frequency         0
Default_Risk                  0
Loan_Approval_Status          0
dtype: int64

In [6]:
df.head()

,Applicant_ID,Gender,Age,Marital_Status,Dependents,Education,Employment_Status,Occupation_Type,Residential_Status,City/Town,...,Loan_Amount_Requested,Loan_Term,Loan_Purpose,Interest_Rate,Loan_Type,Co-Applicant,Bank_Account_History,Transaction_Frequency,Default_Risk,Loan_Approval_Status
0,1,Female,25,Married,2,Graduate,Employed,Business,Own,Urban,...,24535,209,Home,4.27,Secured,Yes,8,20,0.81,1
1,2,Male,36,Married,2,High School,Employed,Business,Own,Suburban,...,8288,33,Home,14.78,Unsecured,Yes,9,9,0.17,0
2,3,Male,43,Single,0,Postgraduate,Self-Employed,Freelancer,Own,Urban,...,10308,159,Vehicle,12.33,Secured,Yes,7,27,0.25,0
3,4,Female,28,Married,0,High School,Self-Employed,Freelancer,Rent,Suburban,...,33937,39,Personal,8.77,Secured,No,9,16,0.27,1
4,5,Female,32,Single,0,Graduate,Employed,Salaried,Rent,Suburban,...,23360,34,Home,9.04,Unsecured,No,1,17,0.32,1


In [7]:
# label encoding (categorical -> numbers)
le = LabelEncoder()
cateforical_cols = ['Gender','Marital_Status','Dependents','Education','Occupation_Type','Loan_Approval_Status','Residential_Status','Loan_Purpose','Loan_Type','Co-Applicant','City/Town','Employment_Status']
for col in cateforical_cols:
    df[col] = le.fit_transform(df[col])

In [8]:
df.head()

,Applicant_ID,Gender,Age,Marital_Status,Dependents,Education,Employment_Status,Occupation_Type,Residential_Status,City/Town,...,Loan_Amount_Requested,Loan_Term,Loan_Purpose,Interest_Rate,Loan_Type,Co-Applicant,Bank_Account_History,Transaction_Frequency,Default_Risk,Loan_Approval_Status
0,1,0,25,1,2,0,0,0,1,2,...,24535,209,1,4.27,0,1,8,20,0.81,1
1,2,1,36,1,2,1,0,0,1,1,...,8288,33,1,14.78,1,1,9,9,0.17,0
2,3,1,43,2,0,2,1,1,1,2,...,10308,159,3,12.33,0,1,7,27,0.25,0
3,4,0,28,1,0,1,1,1,2,1,...,33937,39,2,8.77,0,0,9,16,0.27,1
4,5,0,32,2,0,0,0,3,2,1,...,23360,34,1,9.04,1,0,1,17,0.32,1


In [9]:
# feature  (X). or target (y)
X = df.drop('Loan_Approval_Status',axis=1)
y = df['Loan_Approval_Status']

In [10]:
# Train test split 
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)

(41600, 26)


In [11]:
# best model shape -> random forest 
model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [12]:
# accuracy check 
y_pred = model.predict(X_test)
print(f'accuracy - {accuracy_score(y_test,y_pred)}')
print(f'classification report = {classification_report(y_test,y_pred)}')

accuracy - 0.8556730769230769
classification report =               precision    recall  f1-score   support

           0       0.85      0.71      0.77      3603
           1       0.86      0.93      0.89      6797

    accuracy                           0.86     10400
   macro avg       0.85      0.82      0.83     10400
weighted avg       0.86      0.86      0.85     10400



In [13]:
# use shap
# treeExplainer 
explainer = shap.TreeExplainer(model)

In [ ]:
# shape value calculate 
shape_values = explainer.shap_values(X_test)

In [ ]:
plt.figure(figsize=(10,6))
shap.summary_plot(shape_values[1],X_test,plot_type="bar",'feature_names=X.columns,show=False')
plt.title("Shap feature Importance - loan Approval")
plt.show()

In [ ]:
# https://www.kaggle.com/datasets/arbaaztamboli/loan-approval-dataset